# Necessary preparation using Colab

In [1]:
!apt-get install -y -qq coinor-cbc

Selecting previously unselected package coinor-libcoinutils3v5:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../0-coinor-libcoinutils3v5_2.11.4+repack1-2_amd64.deb ...
Unpacking coinor-libcoinutils3v5:amd64 (2.11.4+repack1-2) ...
Selecting previously unselected package coinor-libosi1v5:amd64.
Preparing to unpack .../1-coinor-libosi1v5_0.108.6+repack1-2_amd64.deb ...
Unpacking coinor-libosi1v5:amd64 (0.108.6+repack1-2) ...
Selecting previously unselected package coinor-libclp1.
Preparing to unpack .../2-coinor-libclp1_1.17.5+repack1-1_amd64.deb ...
Unpacking coinor-libclp1 (1.17.5+repack1-1) ...
Selecting previously unselected package coinor-libcgl1:amd64.
Preparing to unpack .../3-coinor-libcgl1_0.60.3+repack1-3_amd64.deb ...
Unpacking coinor-libcgl1:amd64 (0.60.3+repack1-3) ...
Selecting previously unselected package coinor-libcbc3:amd64.
Preparing to unpack .../4-coinor-libcbc3_2.10.7+ds1-1_amd64.deb ...
Unpacking coinor-libcbc3:

# Preparing the Data


In [3]:
from google.colab import drive
import pandas as pd
import pyomo.environ as pyo

drive.mount('/content/drive')

flights_df = pd.read_csv('/content/drive/MyDrive/flight_pairs.csv', parse_dates=['OutDepTime', 'OutArrTime', 'InDepTime', 'InArrTime'])
crew_df = pd.read_csv('/content/drive/MyDrive/crew.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Incompitable flight pairs (making things easier to satisfy operational requirements)


In [4]:
import itertools
from datetime import timedelta

incomp_pairs = [] # list of tuples
round_pairs_list = flights_df["PairID"].to_list()

for round_A, round_B in itertools.combinations(round_pairs_list, 2):
  row_A = flights_df[flights_df["PairID"] == round_A].iloc[0]
  row_B = flights_df[flights_df["PairID"] == round_B].iloc[0]
  # making sure round_A is before round_B
  if row_A["OutDepTime"] > row_B["OutDepTime"]:
    row_A, row_B = row_B, row_A
  # as per report, incompitable pairs def. -> wait time <= 24 hours
  wait_time = row_B['OutDepTime'] - row_A['InArrTime']
  if wait_time < timedelta( hours = 24):
    incomp_pairs.append((row_A['PairID'],row_B['PairID'] ))
print(f'found {len(incomp_pairs)} incompitable pairs')

found 517 incompitable pairs


# Building the Model

In [5]:
model_t1 = pyo.ConcreteModel("Crew Flight Pair - task 1")

# Sets we iterate over
crew_list = crew_df["CrewID"].to_list()
flights_list = round_pairs_list

# Variables we are trying to control and optimize
# Binary for each crew-flight pair -> each element in the pair is either 0 or 1
model_t1.Assign = pyo.Var(crew_list,flights_list, domain=pyo.Binary)


# Now, the constraints
# we can either use constraint list as per lectures or rule based ones with pyo.Set, it's like using victorization or loops in matrix operations
# Rule-based constraints are better for large scale processing, but we'll stick with the constraintList for now
model_t1.constraints = pyo.ConstraintList()

# Constraint 01 -> the 24 hours wait time (one crew member)
for c in crew_list:
  for f1,f2 in incomp_pairs:
    model_t1.constraints.add(model_t1.Assign[c, f1] +model_t1.Assign[c, f2] <= 1)

# Constraint 02 -> 1P, 2C per pair/flight
pilots = crew_df[crew_df['Role'] == 'Pilot']['CrewID'].tolist()
cabin = crew_df[crew_df['Role'] == 'Cabin']['CrewID'].tolist()
for f in flights_list:
  model_t1.constraints.add(sum(model_t1.Assign[p,f] for p in pilots) == 1)
  model_t1.constraints.add(sum(model_t1.Assign[c,f] for c in cabin) == 2)

# Csonstraint 03 -> aircraft qualification: crew_df["QualifiedA"] & flights_df["OutAircraft"] OR flights_df["InAircraft"]
flight_aircraft_dict = dict(zip(flights_df["PairID"], flights_df["OutAircraft"]))
incomp_aircraft_crew = [] # list of tuples (crew_member, flight)

for index, row in crew_df.iterrows():
    c = row['CrewID']
    qualified_aircrafts = [plane.strip() for plane in row['QualifiedAircraft'].split(',')]
    for f in flights_list:
        flight_plane = flight_aircraft_dict[f]
        # If the plane required for the flight is NOT in the crew member's qualified list
        if flight_plane not in qualified_aircrafts:
            incomp_aircraft_crew.append((c, f))
# Forcing the assignment to equal 0, interesting
for c, f in incomp_aircraft_crew:
    model_t1.constraints.add(model_t1.Assign[c, f] == 0)

# Constraint 04 -> max pairs/flights per month per crew member: crew_df["MaxPairsPerMonth"]
# Calculate outbound and inbound flight durations in hours
flights_df['OutDuration'] = (flights_df['OutArrTime'] - flights_df['OutDepTime']).dt.total_seconds() / 3600.0
flights_df['InDuration'] = (flights_df['InArrTime'] - flights_df['InDepTime']).dt.total_seconds() / 3600.0
# Total actual flying hours for the pair
flights_df['TotalFlightHours'] = flights_df['OutDuration'] + flights_df['InDuration']
# Convert this column into a Python dictionary for extremely fast Pyomo lookup
# Format: {'P001': 8.3, 'P002': 12.5, ...}
flight_hours_dict = dict(zip(flights_df['PairID'], flights_df['TotalFlightHours']))

for index, row in crew_df.iterrows():
    c = row['CrewID']
    role = row['Role']

    max_pairs = row['MaxPairsPerMonth']
    max_hours = 100 if role == 'Pilot' else 150

    model_t1.constraints.add(  sum(model_t1.Assign[c, f] for f in flights_list) <= max_pairs  )
# Constraint 05 -> max hours per month: {p:100, c:150}
    model_t1.constraints.add(  sum(model_t1.Assign[c, f]*flight_hours_dict[f] for f in flights_list) <= max_hours  )




In [6]:
# Objective function
hourly_rate_dict = dict(zip(crew_df['CrewID'], crew_df['HourlyRate']))

# Objective: Minimize total cost (Assign * Hours * HourlyRate)
def obj_rule(model):
    total_cost = sum(
        model.Assign[c, f] * flight_hours_dict[f] * hourly_rate_dict[c]
        for c in crew_list for f in flights_list
    )
    return total_cost
# rule-based objective function
model_t1.Cost = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

# Solving the model

In [7]:
# Solve the model
solver = pyo.SolverFactory("cbc")
solver.options["ratioGAP"] = 0.01  # 1% optimality gap
solver.options["seconds"] = 120    # 2-minute time limit

results = solver.solve(model_t1, tee=True) # tee=True shows the solver's log

Welcome to the CBC MILP Solver 
Version: 2.10.7 
Build Date: Feb 14 2022 

command line - /usr/bin/cbc -ratioGAP 0.01 -seconds 120 -printingOptions all -import /tmp/tmp96p2ojkh.pyomo.lp -stat=1 -solve -solu /tmp/tmp96p2ojkh.pyomo.soln (default strategy 1)
ratioGap was changed from 0 to 0.01
seconds was changed from 1e+100 to 120
Option for printingOptions changed from normal to all
Presolve 47213 (-64402) rows, 12385 (-7615) columns and 130381 (-144034) elements
Statistics for presolved model
Original problem has 20000 integers (20000 of which binary)
Presolved problem has 12385 integers (12385 of which binary)
==== 0 zero objective 226 different
==== absolute objective values 226 different
==== for integers 0 zero objective 226 different
==== for integers absolute objective values 226 different
===== end objective counts


Problem has 47213 rows, 12385 columns (12385 with objective) and 130381 elements
Column breakdown:
0 of type 0.0->inf, 0 of type 0.0->up, 0 of type lo->inf, 
0 of t

# Investigating the solution


In [8]:
import pyomo.environ as pyo
import pandas as pd

# 1. First, verify the solver actually found an optimal solution
if results.solver.status == pyo.SolverStatus.ok and results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print("✅ Solution is Optimal!")

    # Print the Objective Function value (Total Salary Cost)
    print(f"Total Minimum Cost: ${pyo.value(model_t1.Cost):,.2f}\n")

    # 2. Extract the Assignments
    assignment_data = []

    for c in crew_list:
        for f in flights_list:
            # We use > 0.5 instead of == 1 because integer solvers sometimes return values like 0.999999999 due to floating-point precision.
            if pyo.value(model_t1.Assign[c, f]) > 0.5:
                assignment_data.append({'CrewID': c, 'PairID': f})

    # 3. Convert to a DataFrame for investigation
    schedule_df = pd.DataFrame(assignment_data)

    # Merge in the helpful context from your original DataFrames
    schedule_df = schedule_df.merge(crew_df[['CrewID', 'Role', 'HourlyRate']], on='CrewID')
    schedule_df = schedule_df.merge(flights_df[['PairID', 'OutDepTime', 'OutAircraft', 'TotalFlightHours']], on='PairID')

    # Calculate the specific cost of this assignment
    schedule_df['AssignmentCost'] = schedule_df['HourlyRate'] * schedule_df['TotalFlightHours']

    # Sort the dataframe by Flight ID, then by Role to easily check coverage
    schedule_df = schedule_df.sort_values(by=['PairID', 'Role'], ascending=[True, False]).reset_index(drop=True)

    # Display the first few flights to verify
    display(schedule_df.head(15))

elif results.solver.termination_condition == pyo.TerminationCondition.infeasible:
    print("❌ Model is Infeasible! The constraints are contradicting each other.")
else:
    print(f"⚠️ Solver finished with status: {results.solver.status}")

✅ Solution is Optimal!
Total Minimum Cost: $144,580.00



,CrewID,PairID,Role,HourlyRate,OutDepTime,OutAircraft,TotalFlightHours,AssignmentCost
0,PILOT003,P001,Pilot,100,2025-05-15 07:00:00,B737,8.2,820.0
1,CABIN038,P001,Cabin,50,2025-05-15 07:00:00,B737,8.2,410.0
2,CABIN097,P001,Cabin,50,2025-05-15 07:00:00,B737,8.2,410.0
3,PILOT003,P002,Pilot,100,2025-05-17 06:00:00,B737,7.2,720.0
4,CABIN038,P002,Cabin,50,2025-05-17 06:00:00,B737,7.2,360.0
5,CABIN047,P002,Cabin,50,2025-05-17 06:00:00,B737,7.2,360.0
6,PILOT021,P003,Pilot,110,2025-05-02 15:00:00,B737,9.6,1056.0
7,CABIN093,P003,Cabin,50,2025-05-02 15:00:00,B737,9.6,480.0
8,CABIN094,P003,Cabin,50,2025-05-02 15:00:00,B737,9.6,480.0
9,PILOT046,P004,Pilot,100,2025-05-10 08:00:00,A320,9.8,980.0


# Task 2

In [31]:
# TASK 2 MODEL
model_t2 = pyo.ConcreteModel("Crew Flight Pair - task 2")

# Variables
model_t2.Assign = pyo.Var(crew_list, flights_list, domain=pyo.Binary)

model_t2.constraints = pyo.ConstraintList()

In [32]:
# 1. NEW REST CONSTRAINT (48h for pilots only)
incomp_pairs_48 = []

for round_A, round_B in itertools.combinations(round_pairs_list, 2):
    row_A = flights_df[flights_df["PairID"] == round_A].iloc[0]
    row_B = flights_df[flights_df["PairID"] == round_B].iloc[0]

    if row_A["OutDepTime"] > row_B["OutDepTime"]:
        row_A, row_B = row_B, row_A

    wait_time = row_B['OutDepTime'] - row_A['InArrTime']

    if wait_time < timedelta(hours=48):  # NEW RULE
        incomp_pairs_48.append((row_A['PairID'], row_B['PairID']))

# Apply ONLY to pilots
for p in pilots:
    for f1, f2 in incomp_pairs_48:
        model_t2.constraints.add(model_t2.Assign[p, f1] + model_t2.Assign[p, f2] <= 1)

# Cabin crew still 24h
for c in cabin:
    for f1, f2 in incomp_pairs:
        model_t2.constraints.add(model_t2.Assign[c, f1] + model_t2.Assign[c, f2] <= 1)


In [33]:
# 2. COVERAGE CONSTRAINT
for f in flights_list:
    model_t2.constraints.add(sum(model_t2.Assign[p, f] for p in pilots) == 1)
    model_t2.constraints.add(sum(model_t2.Assign[c, f] for c in cabin) == 2)


# 3. QUALIFICATION
for c, f in incomp_aircraft_crew:
    model_t2.constraints.add(model_t2.Assign[c, f] == 0)


# 4. MAX PAIRS + HOURS
for index, row in crew_df.iterrows():
    c = row['CrewID']
    role = row['Role']

    max_pairs = row['MaxPairsPerMonth']
    max_hours = 50 if role == 'Pilot' else 150   # NEW RULE

    model_t2.constraints.add(
        sum(model_t2.Assign[c, f] for f in flights_list) <= max_pairs
    )

    model_t2.constraints.add(
        sum(model_t2.Assign[c, f] * flight_hours_dict[f] for f in flights_list) <= max_hours
    )


In [34]:
# OBJECTIVE (same)
model_t2.Cost = pyo.Objective(
    expr=sum(
        model_t2.Assign[c, f] * flight_hours_dict[f] * hourly_rate_dict[c]
        for c in crew_list for f in flights_list
    ),
    sense=pyo.minimize
)

In [35]:
# Solve
solver = pyo.SolverFactory("cbc")
solver.options["ratioGAP"] = 0.01
solver.options["seconds"] = 120

results_t2 = solver.solve(model_t2, tee=True)

Welcome to the CBC MILP Solver 
Version: 2.10.7 
Build Date: Feb 14 2022 

command line - /usr/bin/cbc -ratioGAP 0.01 -seconds 120 -printingOptions all -import /tmp/tmpevfepcg5.pyomo.lp -stat=1 -solve -solu /tmp/tmpevfepcg5.pyomo.soln (default strategy 1)
ratioGap was changed from 0 to 0.01
seconds was changed from 1e+100 to 120
Option for printingOptions changed from normal to all
Presolve 51301 (-74714) rows, 12385 (-7615) columns and 138557 (-164658) elements
Statistics for presolved model
Original problem has 20000 integers (20000 of which binary)
Presolved problem has 12385 integers (12385 of which binary)
==== 0 zero objective 226 different
==== absolute objective values 226 different
==== for integers 0 zero objective 226 different
==== for integers absolute objective values 226 different
===== end objective counts


Problem has 51301 rows, 12385 columns (12385 with objective) and 138557 elements
Column breakdown:
0 of type 0.0->inf, 0 of type 0.0->up, 0 of type lo->inf, 
0 of t

In [36]:
# Extract PILOT cost only
pilot_cost_t2 = sum(
    pyo.value(model_t2.Assign[p, f]) * flight_hours_dict[f] * hourly_rate_dict[p]
    for p in pilots for f in flights_list
)

pilot_cost_t1 = sum(
    pyo.value(model_t1.Assign[p, f]) * flight_hours_dict[f] * hourly_rate_dict[p]
    for p in pilots for f in flights_list
)

print(f"Pilot Cost Task 1: ${pilot_cost_t1:,.2f}")
print(f"Pilot Cost Task 2: ${pilot_cost_t2:,.2f}")

impact = ((pilot_cost_t2 - pilot_cost_t1) / pilot_cost_t1) * 100
print(f"Impact on Pilot Budget: {impact:.2f}%")

Pilot Cost Task 1: $72,900.00
Pilot Cost Task 2: $74,682.00
Impact on Pilot Budget: 2.44%


# Task 3

In [37]:
# TASK 3 MODEL
model_t3 = pyo.ConcreteModel("Crew Flight Pair - task 3")

# Variables
model_t3.Assign = pyo.Var(crew_list, flights_list, domain=pyo.Binary)
model_t3.Operate = pyo.Var(flights_list, domain=pyo.Binary)

model_t3.constraints = pyo.ConstraintList()

In [38]:
# LINK ASSIGNMENT TO OPERATION
for f in flights_list:
    for c in crew_list:
        model_t3.constraints.add(model_t3.Assign[c, f] <= model_t3.Operate[f])

In [39]:
# REST CONSTRAINTS (48h pilots, 24h cabin)
for p in pilots:
    for f1, f2 in incomp_pairs_48:
        model_t3.constraints.add(model_t3.Assign[p, f1] + model_t3.Assign[p, f2] <= 1)

for c in cabin:
    for f1, f2 in incomp_pairs:
        model_t3.constraints.add(model_t3.Assign[c, f1] + model_t3.Assign[c, f2] <= 1)


In [40]:
# COVERAGE ONLY IF OPERATED
for f in flights_list:
    model_t3.constraints.add(
        sum(model_t3.Assign[p, f] for p in pilots) == model_t3.Operate[f]
    )
    model_t3.constraints.add(
        sum(model_t3.Assign[c, f] for c in cabin) == 2 * model_t3.Operate[f]
    )

# QUALIFICATION
for c, f in incomp_aircraft_crew:
    model_t3.constraints.add(model_t3.Assign[c, f] == 0)


# MAX PAIRS + HOURS (same as Task 2)
for index, row in crew_df.iterrows():
    c = row['CrewID']
    role = row['Role']

    max_pairs = row['MaxPairsPerMonth']
    max_hours = 50 if role == 'Pilot' else 150

    model_t3.constraints.add(
        sum(model_t3.Assign[c, f] for f in flights_list) <= max_pairs
    )

    model_t3.constraints.add(
        sum(model_t3.Assign[c, f] * flight_hours_dict[f] for f in flights_list) <= max_hours
    )

# PILOT BUDGET CONSTRAINT
model_t3.constraints.add(
    sum(
        model_t3.Assign[p, f] * flight_hours_dict[f] * hourly_rate_dict[p]
        for p in pilots for f in flights_list
    ) <= pilot_cost_t1  # fixed from Task 1
)

In [41]:
# OBJECTIVE: MAX REVENUE
revenue_dict = dict(zip(flights_df['PairID'], flights_df['EstimatedRevenue']))

model_t3.Revenue = pyo.Objective(
    expr=sum(model_t3.Operate[f] * revenue_dict[f] for f in flights_list),
    sense=pyo.maximize
)

In [42]:
# Solve
results_t3 = solver.solve(model_t3, tee=True)

Welcome to the CBC MILP Solver 
Version: 2.10.7 
Build Date: Feb 14 2022 

command line - /usr/bin/cbc -ratioGAP 0.01 -seconds 120 -printingOptions all -import /tmp/tmpgmvo6p0v.pyomo.lp -stat=1 -solve -solu /tmp/tmpgmvo6p0v.pyomo.soln (default strategy 1)
ratioGap was changed from 0 to 0.01
seconds was changed from 1e+100 to 120
Option for printingOptions changed from normal to all
 CoinLpIO::readLp(): Maximization problem reformulated as minimization
Coin0009I Switching back to maximization to get correct duals etc
Presolve 63687 (-82329) rows, 12485 (-7615) columns and 166060 (-182355) elements
Statistics for presolved model
Original problem has 20100 integers (20100 of which binary)
Presolved problem has 12485 integers (12485 of which binary)
==== 12385 zero objective 101 different
==== absolute objective values 101 different
==== for integers 12385 zero objective 101 different
==== for integers absolute objective values 101 different
===== end objective counts


Problem has 63687 r

In [43]:
# RESULTS

# Operated flights
operated_flights = [f for f in flights_list if pyo.value(model_t3.Operate[f]) > 0.5]

# Revenue achieved
revenue_obtained = sum(revenue_dict[f] for f in operated_flights)

# Max possible revenue (all flights)
max_revenue = sum(revenue_dict[f] for f in flights_list)

loss_percentage = ((max_revenue - revenue_obtained) / max_revenue) * 100

print(f"Revenue Obtained: ${revenue_obtained:,.2f}")
print(f"Max Possible Revenue: ${max_revenue:,.2f}")
print(f"Revenue Loss %: {loss_percentage:.2f}%")

print(f"Number of flights operated: {len(operated_flights)} / {len(flights_list)}")

Revenue Obtained: $3,854,832.00
Max Possible Revenue: $3,876,806.00
Revenue Loss %: 0.57%
Number of flights operated: 98 / 100
